# Polar Vortex Analysis: Geopotential Height and Wind Fields

This notebook analyzes the geopotential height (Z3) and wind fields (U and V) at two pressure levels (10 hPa and 50 hPa) for two experiments: **nocoupl** and **small**.

For each year and month, daily composite polar maps (in North Polar Stereographic projection) are created with Z3 shown as a filled contour and wind vectors overlaid only at latitude 60° (with half the original density). 

Fixed color scales are set as follows:
- 10 hPa: 26000–34000
- 50 hPa: 17000–22000

The output directory for this analysis is:
```
/home/weiji/restart_exam/20250221/polar_vortex_analysis_G/Z3_wind
```

Time conversion is handled using the snippet below:
```python
time_val = field1.time.values 
if isinstance(time_val, np.ndarray) and time_val.size == 1:
    time_val = time_val.item()
time_val = np.datetime64(time_val)
time_label = np.datetime_as_string(time_val, unit='D')
```

In [ ]:
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import rc
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
import imageio.v2 as imageio

# Set up the output parent directory
output_parent = "/home/weiji/restart_exam/20250221/polar_vortex_analysis_G/Z3_wind"
os.makedirs(output_parent, exist_ok=True)

# Define the years and months to analyze
years = ['0008', '0013', '0014', '0019']
months = ['Feb', 'Mar']

# Define the pressure levels in hPa
pressure_levels = [10, 50]

#########################################
# Function: load_experiment_field_for_var
#########################################
def load_experiment_field_for_var(file_pattern, var_name, lat_min=60, lat_max=90):
    ds = xr.open_mfdataset(file_pattern, concat_dim='member', combine='nested')
    data = ds[var_name]
    data = data.sel(lat=slice(lat_min, lat_max))
    if 'member' in data.dims:
        data = data.mean(dim='member')
    return data

#########################################
# Functions: Load Data for 'nocoupl' and 'small'
#########################################
def load_year_nocouple_field_var(year, month, var_name, lat_min=60, lat_max=90):
    if month == 'Feb':
        file_pattern = f'/home/weiji/restart_exam/nocouple_data/{year}/Feb_NOCOUPL/{year}-02/*.nc'
    elif month == 'Mar':
        file_pattern = f'/home/weiji/restart_exam/nocouple_data/{year}/Mar_NOCOUPL/{year}-03/*.nc'
    else:
        raise ValueError("Invalid month")
    return load_experiment_field_for_var(file_pattern, var_name, lat_min, lat_max)

def load_year_small_pert_field_var(year, month, var_name, lat_min=60, lat_max=90):
    if month == 'Feb':
        file_pattern = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}/Feb/BWCN.e122.f19_g16.002.{year}-02.*.nc'
    elif month == 'Mar':
        file_pattern = f'/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.002_{year}/Mar/BWCN.e122.f19_g16.002.{year}-03.*.nc'
    else:
        raise ValueError("Invalid month")
    return load_experiment_field_for_var(file_pattern, var_name, lat_min, lat_max)

#########################################
# Function: plot_composite_daily_maps_Z3_wind
#########################################
def plot_composite_daily_maps_Z3_wind(exp1_name, Z3_data1, U_data1, V_data1,
                                      exp2_name, Z3_data2, U_data2, V_data2,
                                      output_dir, pressure_level, frame_duration=500):
    os.makedirs(output_dir, exist_ok=True)
    rc('font', **{'family': 'sans-serif', 'sans-serif': ['Helvetica']})
    rc('text', usetex=False)

    # 根据不同压力层设置固定色标
    if pressure_level == 10:
        vmin, vmax = 26000, 34000
    elif pressure_level == 50:
        vmin, vmax = 17000, 22000
    else:
        vmin, vmax = None, None

    target_pressure = pressure_level * 100

    Z3_data1_level = Z3_data1.sel(plev=target_pressure, method='nearest')
    U_data1_level = U_data1.sel(plev=target_pressure, method='nearest')
    V_data1_level = V_data1.sel(plev=target_pressure, method='nearest')
    
    Z3_data2_level = Z3_data2.sel(plev=target_pressure, method='nearest')
    U_data2_level = U_data2.sel(plev=target_pressure, method='nearest')
    V_data2_level = V_data2.sel(plev=target_pressure, method='nearest')
    
    n_frames = min(Z3_data1_level.time.size, Z3_data2_level.time.size)
    
    for i in range(n_frames):
        # Extract time slices for both experiments
        Z3_1 = Z3_data1_level.isel(time=i)
        U_1 = U_data1_level.isel(time=i)
        V_1 = V_data1_level.isel(time=i)
        
        Z3_2 = Z3_data2_level.isel(time=i)
        U_2 = U_data2_level.isel(time=i)
        V_2 = V_data2_level.isel(time=i)
        
        Z3_1_vals, lons = add_cyclic_point(Z3_1.values, coord=Z3_1.lon.values)
        Z3_2_vals, _ = add_cyclic_point(Z3_2.values, coord=Z3_2.lon.values)
        
        # Process time label using the provided snippet
        time_val = Z3_1.time.values
        if isinstance(time_val, np.ndarray) and time_val.size == 1:
            time_val = time_val.item()
        time_val = np.datetime64(time_val)
        time_label = np.datetime_as_string(time_val, unit='D')
        
        fig, axes = plt.subplots(1, 2, subplot_kw={'projection': ccrs.NorthPolarStereo()}, figsize=(16,8))
        for ax in axes:
            ax.set_extent([-180, 180, 60, 90], ccrs.PlateCarree())
            ax.coastlines()
            ax.add_feature(cfeature.LAND, facecolor='lightgray')
            gl = ax.gridlines(draw_labels=True, linestyle='--', color='gray')
            gl.xlabels_top = False
            gl.ylabels_right = False
        
        # Left panel: full field Z3 contour and overlaid wind vectors only at lat=60°
        cf1 = axes[0].contourf(lons, Z3_1.lat, Z3_1_vals, levels=np.linspace(vmin, vmax, 21),
                               cmap='viridis', extend='both', transform=ccrs.PlateCarree())
        
        # 提取北纬60°附近的索引
        lat_idx_1 = np.argmin(np.abs(U_1.lat.values - 60))
        # 对经度方向密度降低一半
        U_1_subset = U_1.values[lat_idx_1, ::2]
        V_1_subset = V_1.values[lat_idx_1, ::2]
        lon_subset = U_1.lon.values[::2]
        # 在左侧图中绘制风矢量，scale 调整为700
        axes[0].quiver(lon_subset, np.full_like(lon_subset, U_1.lat.values[lat_idx_1]),
                       U_1_subset, V_1_subset, transform=ccrs.PlateCarree(), scale=700)
        axes[0].set_title(f"{exp1_name} Z3 at {pressure_level} hPa\nDate: {time_label}", fontsize=14)
        
        # Right panel: 同样处理
        cf2 = axes[1].contourf(lons, Z3_2.lat, Z3_2_vals, levels=np.linspace(vmin, vmax, 21),
                               cmap='viridis', extend='both', transform=ccrs.PlateCarree())
        lat_idx_2 = np.argmin(np.abs(U_2.lat.values - 60))
        U_2_subset = U_2.values[lat_idx_2, ::2]
        V_2_subset = V_2.values[lat_idx_2, ::2]
        lon_subset_2 = U_2.lon.values[::2]
        axes[1].quiver(lon_subset_2, np.full_like(lon_subset_2, U_2.lat.values[lat_idx_2]),
                       U_2_subset, V_2_subset, transform=ccrs.PlateCarree(), scale=700)
        axes[1].set_title(f"{exp2_name} Z3 at {pressure_level} hPa\nDate: {time_label}", fontsize=14)
        
        cbar = fig.colorbar(cf2, ax=axes, orientation='horizontal', fraction=0.05, pad=0.08)
        cbar.set_label(f"Geopotential Height (m) at {pressure_level} hPa", fontsize=12)
        
        out_fname = os.path.join(output_dir, f"frame_{i+1:03d}.png")
        plt.savefig(out_fname, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f"Saved frame {i+1} to {out_fname}")
    
    files = sorted([f for f in os.listdir(output_dir) if f.endswith('.png')])
    if files:
        images = []
        for fname in files:
            fpath = os.path.join(output_dir, fname)
            try:
                images.append(imageio.imread(fpath))
            except Exception as e:
                print(f"Error reading {fpath}: {e}")
        gif_fname = os.path.join(output_dir, f"composite_Z3_wind_{pressure_level}hPa.gif")
        imageio.mimsave(gif_fname, images, duration=frame_duration/1000)
        print(f"Saved GIF: {gif_fname}")
    else:
        print(f"No PNG files found in {output_dir}!")

#########################################
# Main processing loop
#########################################
for year in years:
    for month in months:
        print(f"Processing Year: {year}, Month: {month}")
        Z3_nocoup = load_year_nocouple_field_var(year, month, 'Z3')
        U_nocoup = load_year_nocouple_field_var(year, month, 'U')
        V_nocoup = load_year_nocouple_field_var(year, month, 'V')
        
        Z3_small = load_year_small_pert_field_var(year, month, 'Z3')
        U_small = load_year_small_pert_field_var(year, month, 'U')
        V_small = load_year_small_pert_field_var(year, month, 'V')
        
        for p in pressure_levels:
            print(f"Processing pressure level: {p} hPa")
            out_dir = os.path.join(output_parent, f"{year}_{month}_{p}hPa")
            os.makedirs(out_dir, exist_ok=True)
            plot_composite_daily_maps_Z3_wind("nocoupl", Z3_nocoup, U_nocoup, V_nocoup,
                                              "small", Z3_small, U_small, V_small,
                                              out_dir, p, frame_duration=500)

print("All composite GIFs for Z3 and wind fields have been generated!")
